In [1]:
import pandas
import os
import re
import numpy as np
import pandas as pd
# Add relative import path
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator

from ba138_spd_zeeman_levels_optical_bloch_linsolve_v2 import ba138_spd_zeeman_linsolve_v2


In [2]:
def set_size(width_pt = 760.53693, fraction=1, subplots=(1, 1)):
    """Set figure dimensions to avoid scaling in LaTeX."""
    # Width of figure (in pts)
    fig_width_pt = width_pt * fraction
    # Convert from pt to inches
    inches_per_pt = 1 / 72.27

    # Golden ratio to set aesthetic height
    golden_ratio = (5**.5 - 1) / 2

    # Figure width in inches
    fig_width_in = fig_width_pt * inches_per_pt
    # Figure height in inches
    fig_height_in = fig_width_in * golden_ratio * (subplots[0] / subplots[1])

    return (fig_width_in, fig_height_in)

plt.rcParams['figure.dpi'] = 150  # Set higher DPI for better quality
plt.rcParams['font.size'] = 14  # Set font size for better readability
# Enable LaTeX rendering
# plt.rcParams.update({
#     "text.usetex": False,
#     "font.family": "CMU Sans Serif",
#     "font.serif": ["Medium"],
# })

In [ ]:
def run_delta_scan(Omega_S, Omega_D, Delta_S_array=None, Delta_D_array=None):
    """
    Scans over S-P and D-P detunings and computes steady-state P state population.

    Parameters:
        Omega_S: Rabi frequency of S-P transition (rad*MHz)
        Omega_D: Rabi frequency of D-P transition (rad*MHz)

    Returns:
        Delta_S_array: Array of S-P detunings (MHz)
        Delta_D_array: Array of D-P detunings (MHz)
        sigma_PP_end: 2D array of P state populations
    """
    if Delta_S_array is None:
        Delta_S_array = np.arange(-30, 30, 1)  # Range of frequency detunings for S-P in MHz
    if Delta_D_array is None:
        Delta_D_array = np.arange(-30, 30, 1)  # Range of frequency detunings for D-P in MHz
    sigma_PP_end = np.full((len(Delta_S_array), len(Delta_D_array)), np.nan)

    for hh in range(len(Delta_S_array)):
        for hhh in range(len(Delta_D_array)):
            Delta_S = 2 * np.pi * Delta_S_array[hh]  # Scale frequency by 2*pi
            Delta_D = 2 * np.pi * Delta_D_array[hhh]  # Scale frequency by 2*pi
            sigma_PP, cp0, cp0_c, cp1, cp1_c = ba138_spd_zeeman_linsolve_v2(Delta_S, Delta_D, Omega_S, Omega_D, theta_s=0, theta_d=0, B_gauss=1.8)
            sigma_PP_end[hh, hhh] = sigma_PP.real

    # Analytical form may have values divided by zero, giving NaN. Replace with zero.
    sigma_PP_end[np.isnan(sigma_PP_end)] = 0

    return Delta_S_array, Delta_D_array, sigma_PP_end

def run_Rabi_scan(Omega_S_array=None, Omega_D_array=None, Delta_S=0, Delta_D=0):
    """
    Scans over S-P and D-P Rabi frequencies and computes steady-state P state population.

    Parameters:
        Omega_S_array: Array of Rabi frequencies for S-P transition (rad*MHz)
        Omega_D_array: Array of Rabi frequencies for D-P transition (rad*MHz)
        Delta_S: Detuning for S-P transition (MHz)
        Delta_D: Detuning for D-P transition (MHz)
    Returns:
        Omega_S_array: Array of Rabi frequencies for S-P transition (rad*MHz)
        Omega_D_array: Array of Rabi frequencies for D-P transition (rad*MHz)
        sigma_PP_end: 2D array of P state populations
    """
    if Omega_S_array is None:
        Omega_S_array = np.arange(0, 20, 0.5) * 2 * np.pi  # Rabi frequencies for S-P in rad*MHz
    if Omega_D_array is None:
        Omega_D_array = np.arange(0, 20, 0.5) * 2 * np.pi  # Rabi frequencies for D-P in rad*MHz
    sigma_PP_end = np.full((len(Omega_S_array), len(Omega_D_array)), np.nan)

    for hh in range(len(Omega_S_array)):
        for hhh in range(len(Omega_D_array)):
            Omega_S = Omega_S_array[hh]  # Rabi frequency for S-P
            Omega_D = Omega_D_array[hhh]  # Rabi frequency for D-P
            sigma_PP, cp0, cp0_c, cp1, cp1_c = ba138_spd_zeeman_linsolve_v2(Delta_S, Delta_D, Omega_S, Omega_D, theta_s=0, theta_d=0, B_gauss=1.8)
            sigma_PP_end[hh, hhh] = sigma_PP.real

    return Omega_S_array, Omega_D_array, sigma_PP_end

In [ ]:
Omega_S = 2 * np.pi * 10
Omega_D = 2 * np.pi * 10
Delta_S_array, Delta_D_array, sigma_PP_end = run_delta_scan(Omega_S, Omega_D)

fig, ax = plt.subplots(figsize=set_size(), constrained_layout=True)
mesh = ax.pcolormesh(Delta_D_array, Delta_S_array, sigma_PP_end, shading='auto')
ax.set_xlabel('Delta_D (MHz)')
ax.set_ylabel('Delta_S (MHz)')
ax.set_title('P state population vs detunings')
plt.colorbar(mappable=mesh, ax=ax, label='P state population')
plt.show()

## Global maximum of sigma_PP over (Omega_S, Omega_D, Delta_S, Delta_D)

The analytical expression is cheap and vectorizable, so instead of alternating 2D scans (which can stall on ridges/saddles), we:

1. Evaluate sigma_PP on a coarse 4D grid covering the full bounds using NumPy broadcasting (one vectorized call).
2. Zoom: shrink the search box around the current argmax by a fixed factor and resample, re-centering on the new argmax each iteration.
3. Polish: hand the best grid point to `scipy.optimize.minimize` (L-BFGS-B) for sub-grid precision.

This converges geometrically in each dimension and avoids the coordinate-descent trap.

In [ ]:
from scipy.optimize import minimize


def sigma_PP_vec(Delta_S, Delta_D, Omega_S, Omega_D):
    """Vectorized sigma_PP.real. Inputs broadcast against each other.
    NaNs (e.g. Delta_S == Delta_D) are mapped to -inf so they lose the argmax."""
    with np.errstate(divide='ignore', invalid='ignore'):
        val = ba_spd_levels_optical_bloch_analytical(Delta_S, Delta_D, Omega_S, Omega_D).real
    return np.where(np.isfinite(val), val, -np.inf)


def find_global_max_sigma_PP(
    bounds=None,
    n_grid=21,
    n_refine=6,
    zoom=0.35,
    polish=True,
    verbose=False,
):
    """Adaptive coarse-to-fine 4D grid search for the global max of sigma_PP.real.

    Parameters
    ----------
    bounds : dict with keys 'Omega_S', 'Omega_D', 'Delta_S', 'Delta_D'. Each
             maps to (lo, hi) in rad*MHz. Defaults to a physics-reasonable box.
    n_grid : grid points per dimension (total cost ~ n_grid**4 per iteration).
    n_refine : number of zoom iterations.
    zoom : shrink factor for the search box per iteration (0 < zoom < 1).
    polish : run L-BFGS-B from the best grid point for sub-grid precision.

    Returns
    -------
    dict with the optimum, history of zoom steps, and the polished result.
    """
    if bounds is None:
        bounds = {
            'Omega_S': (0.5 * 2 * np.pi, 100.0 * 2 * np.pi),
            'Omega_D': (0.5 * 2 * np.pi, 100.0 * 2 * np.pi),
            'Delta_S': (-50.0 * 2 * np.pi, 50.0 * 2 * np.pi),
            'Delta_D': (-50.0 * 2 * np.pi, 50.0 * 2 * np.pi),
        }
    keys = ('Omega_S', 'Omega_D', 'Delta_S', 'Delta_D')
    lo = np.array([bounds[k][0] for k in keys], dtype=float)
    hi = np.array([bounds[k][1] for k in keys], dtype=float)
    global_lo, global_hi = lo.copy(), hi.copy()

    history = []
    best_x = None
    best_val = -np.inf

    for it in range(n_refine):
        axes = [np.linspace(lo[i], hi[i], n_grid) for i in range(4)]
        # Broadcast shapes so sigma_PP_vec evaluates on the full 4D grid in one call.
        Omega_S = axes[0].reshape(n_grid, 1, 1, 1)
        Omega_D = axes[1].reshape(1, n_grid, 1, 1)
        Delta_S = axes[2].reshape(1, 1, n_grid, 1)
        Delta_D = axes[3].reshape(1, 1, 1, n_grid)
        grid = sigma_PP_vec(Delta_S, Delta_D, Omega_S, Omega_D)

        idx = np.unravel_index(np.argmax(grid), grid.shape)
        x = np.array([axes[d][idx[d]] for d in range(4)])
        val = float(grid[idx])
        history.append({'iter': it, 'x': x.copy(), 'value': val,
                        'box_lo': lo.copy(), 'box_hi': hi.copy()})
        if val > best_val:
            best_val, best_x = val, x.copy()
        if verbose:
            print(f"iter {it}: max={val:.6f} at {dict(zip(keys, x))}")

        # Shrink the box around x, clip to original bounds.
        half = 0.5 * zoom * (hi - lo)
        lo = np.maximum(global_lo, x - half)
        hi = np.minimum(global_hi, x + half)

    result = {'x': best_x, 'value': best_val, 'history': history,
              'keys': keys, 'bounds': bounds}

    if polish:
        def neg_obj(x):
            return -sigma_PP_vec(x[2], x[3], x[0], x[1])
        res = minimize(
            neg_obj, x0=best_x, method='L-BFGS-B',
            bounds=[bounds[k] for k in keys],
            options={'ftol': 1e-12, 'gtol': 1e-10},
        )
        if -res.fun > best_val:
            result['x'] = res.x
            result['value'] = float(-res.fun)
        result['polish'] = res

    return result

In [ ]:
result = find_global_max_sigma_PP(n_grid=21, n_refine=8, zoom=0.35, verbose=True)

x = result['x']
print("\nGlobal maximum of sigma_PP.real:")
print(f"  value   = {result['value']:.6f}")
print(f"  Omega_S = {x[0]:.4f} rad*MHz  ({x[0]/(2*np.pi):.4f} MHz)")
print(f"  Omega_D = {x[1]:.4f} rad*MHz  ({x[1]/(2*np.pi):.4f} MHz)")
print(f"  Delta_S = {x[2]:.4f} rad*MHz  ({x[2]/(2*np.pi):.4f} MHz)")
print(f"  Delta_D = {x[3]:.4f} rad*MHz  ({x[3]/(2*np.pi):.4f} MHz)")

In [ ]:
DS_array = np.linspace(x[2] - 20 * 2 * np.pi, x[2] + 20 * 2 * np.pi, 100)
DD_array = np.linspace(x[3] - 20 * 2 * np.pi, x[3] + 20 * 2 * np.pi, 100)

print("\nZoomed scan around the optimum:")
print(DS_array/(2*np.pi))
print(DD_array/(2*np.pi))

In [ ]:
# Sanity-check: cross-section through the optimum in the (Delta_S, Delta_D) plane.
Delta_S_arr, Delta_D_arr, rho_pp = run_delta_scan(x[0], x[1], Delta_S_array=np.linspace(x[2]/ (2 * np.pi) - 80, np.amin([x[2]/ (2 * np.pi) + 80, 0]) , 100),
                                                  Delta_D_array=np.linspace(x[3]/ (2 * np.pi) - 80, x[3]/ (2 * np.pi) + 80, 100))

In [ ]:
fig, ax = plt.subplots(figsize=set_size(), constrained_layout=True)
mesh = ax.pcolormesh(Delta_D_arr, Delta_S_arr, rho_pp, shading='auto')
ax.axhline(x[2] / (2 * np.pi), color='red', lw=0.8, ls='--')
ax.axvline(x[3] / (2 * np.pi), color='red', lw=0.8, ls='--')
ax.set_xlabel(r'$\Delta_D$ (MHz)')
ax.set_ylabel(r'$\Delta_S$ (MHz)')
ax.set_title(fr"$\rho_{{PP}}$ at $\Omega_S=${x[0]/(2*np.pi):.2f}, $\Omega_D=${x[1]/(2*np.pi):.2f} MHz")
plt.colorbar(mesh, ax=ax, label='P state population')
plt.show()